In [1]:
import subprocess, torch, platform, sys
print("python:", sys.version)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
print(subprocess.check_output(["nvidia-smi"]).decode("utf-8")[:1200])

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
capability: (12, 0)
Sat Feb 28 23:35:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       3MiB /  9788

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
import glob
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

PARQUET_DIR = Path("/content/drive/MyDrive/hi_preprocessed")  # <-- your folder
SAMPLE_N = 50_000_000
SEED = 42
rng = np.random.default_rng(SEED)

SAMPLE_OUT_DIR = PARQUET_DIR.parent / f"{PARQUET_DIR.name}_sample_50m"
SAMPLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

files = sorted(glob.glob(str(PARQUET_DIR / "*.parquet")))
if not files:
    raise FileNotFoundError(f"No parquet files in {PARQUET_DIR}")

print("PARQUET_DIR:", PARQUET_DIR)
print("Num files:", len(files))
print("Sample out:", SAMPLE_OUT_DIR)

# count rows from metadata (fast)
total_rows = 0
for f in files:
    total_rows += pq.ParquetFile(f).metadata.num_rows
print("Total rows:", total_rows)

p = min(1.0, SAMPLE_N / max(total_rows, 1))
print("Sampling probability p =", p)

shard_rows = 2_000_000
shard_id = 0
kept = 0
buf_tables = []
buf_rows = 0

def flush_shard():
    global shard_id, buf_tables, buf_rows
    if not buf_tables:
        return
    table = pa.concat_tables(buf_tables, promote_options="default")
    out_path = SAMPLE_OUT_DIR / f"part_{shard_id:04d}.parquet"
    pq.write_table(table, out_path, compression="snappy")
    print(f"Wrote {table.num_rows:,} rows -> {out_path.name}")
    shard_id += 1
    buf_tables = []
    buf_rows = 0

for f in files:
    if kept >= SAMPLE_N:
        break
    pf = pq.ParquetFile(f)
    for batch in pf.iter_batches(batch_size=250_000):
        if kept >= SAMPLE_N:
            break
        n = batch.num_rows
        mask = rng.random(n) < p
        if not mask.any():
            continue
        btab = pa.Table.from_batches([batch]).filter(pa.array(mask))
        rem = SAMPLE_N - kept
        if btab.num_rows > rem:
            btab = btab.slice(0, rem)
        buf_tables.append(btab)
        buf_rows += btab.num_rows
        kept += btab.num_rows
        if buf_rows >= shard_rows:
            flush_shard()

flush_shard()
print("Sample kept:", kept)
print("Sample folder:", SAMPLE_OUT_DIR)

PARQUET_DIR: /content/drive/MyDrive/hi_preprocessed
Num files: 128
Sample out: /content/drive/MyDrive/hi_preprocessed_sample_50m
Total rows: 179702229
Sampling probability p = 0.27823806236705056
Wrote 2,068,790 rows -> part_0000.parquet
Wrote 2,019,809 rows -> part_0001.parquet
Wrote 2,011,464 rows -> part_0002.parquet
Wrote 2,035,765 rows -> part_0003.parquet
Wrote 2,034,016 rows -> part_0004.parquet
Wrote 2,030,213 rows -> part_0005.parquet
Wrote 2,036,332 rows -> part_0006.parquet
Wrote 2,030,972 rows -> part_0007.parquet
Wrote 2,035,316 rows -> part_0008.parquet
Wrote 2,068,369 rows -> part_0009.parquet
Wrote 2,034,506 rows -> part_0010.parquet
Wrote 2,033,736 rows -> part_0011.parquet
Wrote 2,032,786 rows -> part_0012.parquet
Wrote 2,034,784 rows -> part_0013.parquet
Wrote 2,033,311 rows -> part_0014.parquet
Wrote 2,032,963 rows -> part_0015.parquet
Wrote 2,067,906 rows -> part_0016.parquet
Wrote 2,032,902 rows -> part_0017.parquet
Wrote 2,035,011 rows -> part_0018.parquet
Wrote 

In [4]:
import glob
import pyarrow.parquet as pq
import pandas as pd

SAMPLE_OUT_DIR = "/content/drive/MyDrive/hi_preprocessed_sample_50m"  # should match what Cell 2 printed
parts = sorted(glob.glob(f"{SAMPLE_OUT_DIR}/part_*.parquet"))
print("Num parts:", len(parts))

dfs, rows = [], 0
for i, p in enumerate(parts):
    dfi = pq.read_table(p).to_pandas()
    dfs.append(dfi)
    rows += len(dfi)
    if (i+1) % 5 == 0:
        print(f"Loaded {i+1}/{len(parts)} | rows so far: {rows:,}")

df = pd.concat(dfs, ignore_index=True)
print("FINAL df shape:", df.shape)
df.head()

Num parts: 25
Loaded 5/25 | rows so far: 10,169,844
Loaded 10/25 | rows so far: 20,371,046
Loaded 15/25 | rows so far: 30,540,169
Loaded 20/25 | rows so far: 40,743,288
Loaded 25/25 | rows so far: 50,000,000
FINAL df shape: (50000000, 21)


,Timestamp,From Bank,Account,To Bank,Account_1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,...,Log Amount Received,_ts,tx_hour,tx_dow,tx_month,tx_day,tx_date,tx_is_weekend,tx_hour_sin,tx_hour_cos
0,2022/08/01 00:02,1208,80010E650,20,80010EA30,45868454.0,US Dollar,45868454.0,US Dollar,Cheque,...,17.641288,2022-08-01 00:02:00,0,0,8,1,2022-08-01,0,0.0,1.0
1,2022/08/01 00:09,11,80010E600,11,80010E600,16.33,US Dollar,16.33,US Dollar,Reinvestment,...,2.852439,2022-08-01 00:09:00,0,0,8,1,2022-08-01,0,0.0,1.0
2,2022/08/01 00:07,0,8009B22F0,0,8009B22F0,121.98,US Dollar,121.98,US Dollar,Reinvestment,...,4.812022,2022-08-01 00:07:00,0,0,8,1,2022-08-01,0,0.0,1.0
3,2022/08/01 00:22,0,800815DE0,0,800815DE0,23.4,US Dollar,23.4,US Dollar,Reinvestment,...,3.194583,2022-08-01 00:22:00,0,0,8,1,2022-08-01,0,0.0,1.0
4,2022/08/01 00:20,1315,8009BDC30,125106,816DF5400,165.63,US Dollar,165.63,US Dollar,Credit Card,...,5.115776,2022-08-01 00:20:00,0,0,8,1,2022-08-01,0,0.0,1.0


In [5]:
import numpy as np

Y_COL = "Is Laundering"

# DOW one-hot from existing tx_dow
dow = df["tx_dow"].astype(int).clip(0, 6).to_numpy()
dow_oh = np.eye(7, dtype=np.float32)[dow]
for i in range(7):
    df[f"dow_{i}"] = dow_oh[:, i]

feat_cols = [
    "Log Amount Received", "tx_hour_sin", "tx_hour_cos",
    "tx_is_weekend", "tx_hour",
    "dow_0","dow_1","dow_2","dow_3","dow_4","dow_5","dow_6"
]

X_edge = df[feat_cols].to_numpy(dtype=np.float32)
y_edge = df[Y_COL].to_numpy(dtype=np.int64)

print("X_edge:", X_edge.shape, "| y:", y_edge.shape, "| pos_rate:", y_edge.mean())
print("Feature cols:", feat_cols)

X_edge: (50000000, 12) | y: (50000000,) | pos_rate: 0.00126038
Feature cols: ['Log Amount Received', 'tx_hour_sin', 'tx_hour_cos', 'tx_is_weekend', 'tx_hour', 'dow_0', 'dow_1', 'dow_2', 'dow_3', 'dow_4', 'dow_5', 'dow_6']


In [6]:
import numpy as np
import torch

SEED = 42
rng = np.random.default_rng(SEED)

SRC_COL = "Account"
DST_COL = "Account_1"

HASH_BUCKETS = 5_000_000  # tune (2M–10M)

def fnv1a_hash(x) -> int:
    h = 1469598103934665603
    for b in str(x).encode("utf-8"):
        h ^= b
        h = (h * 1099511628211) & 0xFFFFFFFFFFFFFFFF
    return h

src_ids = np.fromiter((fnv1a_hash(v) % HASH_BUCKETS for v in df[SRC_COL].values),
                      dtype=np.int64, count=len(df))
dst_ids = np.fromiter((fnv1a_hash(v) % HASH_BUCKETS for v in df[DST_COL].values),
                      dtype=np.int64, count=len(df))

E = len(src_ids)
perm = rng.permutation(E)
n_tr = int(0.90 * E)
n_va = int(0.05 * E)

train_e = perm[:n_tr]
val_e   = perm[n_tr:n_tr+n_va]
test_e  = perm[n_tr+n_va:]

print("Edges:", E)
print("Split sizes:", {"train": len(train_e), "val": len(val_e), "test": len(test_e)})

src_t = torch.from_numpy(src_ids)          # CPU
dst_t = torch.from_numpy(dst_ids)          # CPU
y_t   = torch.from_numpy(y_edge).float()   # CPU

Edges: 50000000
Split sizes: {'train': 45000000, 'val': 2500000, 'test': 2500000}


In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
rng = np.random.default_rng(SEED)

src_np = src_t.numpy().astype(np.int32, copy=False)
dst_np = dst_t.numpy().astype(np.int32, copy=False)
y_np   = y_t.numpy().astype(np.int8, copy=False)

# CSR build (undirected)
u = np.concatenate([src_np, dst_np], axis=0)
v = np.concatenate([dst_np, src_np], axis=0)

order = np.argsort(u, kind="mergesort")
u_sorted = u[order]
v_sorted = v[order]

deg = np.bincount(u_sorted, minlength=HASH_BUCKETS).astype(np.int32)
offsets = np.empty(HASH_BUCKETS + 1, dtype=np.int64)
offsets[0] = 0
np.cumsum(deg, out=offsets[1:])

neighbors = v_sorted.astype(np.int32, copy=False)

print("CSR ready:",
      "| nodes", HASH_BUCKETS,
      "| undirected edges", neighbors.shape[0],
      "| avg deg", float(deg.mean()),
      "| neighbors GB", neighbors.nbytes/1e9)

# balanced edge sampling
pos_idx = np.where(y_np == 1)[0]
neg_idx = np.where(y_np == 0)[0]
print("Pos edges:", len(pos_idx), "Neg edges:", len(neg_idx), "pos_rate:", y_np.mean())

def sample_edge_batch(batch_edges=200_000):
    half = batch_edges // 2
    pe = rng.choice(pos_idx, size=half, replace=len(pos_idx) < half)
    ne = rng.choice(neg_idx, size=batch_edges - half, replace=False)
    e = np.concatenate([pe, ne], axis=0)
    rng.shuffle(e)
    return e

def sample_neighbors(seed_nodes: np.ndarray, fanout: int):
    seed_nodes = seed_nodes.astype(np.int32, copy=False)
    src_list, dst_list = [], []
    for n in seed_nodes:
        start = offsets[n]
        end = offsets[n + 1]
        d = end - start
        if d <= 0:
            continue
        if d <= fanout:
            nbrs = neighbors[start:end]
        else:
            idx = rng.integers(0, d, size=fanout, endpoint=False)
            nbrs = neighbors[start + idx]
        src_list.append(np.full(len(nbrs), n, dtype=np.int32))
        dst_list.append(nbrs.astype(np.int32, copy=False))
    if not src_list:
        return np.empty(0, np.int32), np.empty(0, np.int32)
    return np.concatenate(src_list), np.concatenate(dst_list)

def map_global_to_local(local_nodes: np.ndarray, arr: np.ndarray) -> np.ndarray:
    return np.searchsorted(local_nodes, arr).astype(np.int64)

class SAGEConvMean(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_nei  = nn.Linear(in_dim, out_dim)

    def forward(self, x, src_idx, dst_idx, num_nodes):
        agg = x.new_zeros((num_nodes, x.size(1)))
        cnt = x.new_zeros((num_nodes, 1))
        agg.index_add_(0, src_idx, x[dst_idx])
        cnt.index_add_(0, src_idx, x.new_ones((dst_idx.numel(), 1)))
        agg = agg / cnt.clamp_min(1.0)
        return F.relu(self.lin_self(x) + self.lin_nei(agg))

class GraphSAGEEdgeGNN(nn.Module):
    def __init__(self, num_nodes, emb_dim=64, hid_dim=64, out_dim=64, edge_feat_dim=12):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, emb_dim)
        self.sage1 = SAGEConvMean(emb_dim, hid_dim)
        self.sage2 = SAGEConvMean(hid_dim, out_dim)
        self.edge_mlp = nn.Sequential(
            nn.Linear(out_dim * 3 + edge_feat_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward_subgraph(self, local_nodes, e1_src, e1_dst, e2_src, e2_dst):
        x0 = self.emb(local_nodes)
        x1 = self.sage1(x0, e1_src, e1_dst, local_nodes.size(0))
        x2 = self.sage2(x1, e2_src, e2_dst, local_nodes.size(0))
        return x2

    def edge_logits(self, h, src_local, dst_local, edge_feat):
        hs = h[src_local]
        hd = h[dst_local]
        z = torch.cat([hs, hd, hs * hd, edge_feat], dim=1)
        return self.edge_mlp(z).squeeze(1)

model = GraphSAGEEdgeGNN(num_nodes=HASH_BUCKETS, edge_feat_dim=X_edge.shape[1]).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)

fanout1, fanout2 = 15, 10
BATCH_EDGES = 200_000
STEPS_PER_EPOCH = 10
EPOCHS = 2

X_edge_cpu = X_edge  # numpy float32

def train_step(edge_ids):
    s = src_np[edge_ids]
    d = dst_np[edge_ids]
    seed = np.unique(np.concatenate([s, d], axis=0)).astype(np.int32)

    e1_s, e1_d = sample_neighbors(seed, fanout1)
    hop1 = np.unique(np.concatenate([seed, e1_d], axis=0)).astype(np.int32)
    e2_s, e2_d = sample_neighbors(hop1, fanout2)
    local_nodes = np.unique(np.concatenate([hop1, e2_d], axis=0)).astype(np.int32)

    e1_src_l = map_global_to_local(local_nodes, e1_s)
    e1_dst_l = map_global_to_local(local_nodes, e1_d)
    e2_src_l = map_global_to_local(local_nodes, e2_s)
    e2_dst_l = map_global_to_local(local_nodes, e2_d)
    src_l = map_global_to_local(local_nodes, s)
    dst_l = map_global_to_local(local_nodes, d)

    local_nodes_t = torch.from_numpy(local_nodes.astype(np.int64)).to(device, non_blocking=True)
    e1_src_t = torch.from_numpy(e1_src_l).to(device)
    e1_dst_t = torch.from_numpy(e1_dst_l).to(device)
    e2_src_t = torch.from_numpy(e2_src_l).to(device)
    e2_dst_t = torch.from_numpy(e2_dst_l).to(device)
    src_tl = torch.from_numpy(src_l).to(device)
    dst_tl = torch.from_numpy(dst_l).to(device)

    yb = torch.from_numpy(y_np[edge_ids].astype(np.float32)).to(device)
    xb = torch.from_numpy(X_edge_cpu[edge_ids]).to(device)

    model.train()
    opt.zero_grad(set_to_none=True)
    h = model.forward_subgraph(local_nodes_t, e1_src_t, e1_dst_t, e2_src_t, e2_dst_t)
    logits = model.edge_logits(h, src_tl, dst_tl, xb)
    loss = F.binary_cross_entropy_with_logits(logits, yb)
    loss.backward()
    opt.step()
    return float(loss.item()), int(yb.sum()), len(yb)

for epoch in range(1, EPOCHS + 1):
    losses = []
    for step in range(STEPS_PER_EPOCH):
        edge_ids = sample_edge_batch(BATCH_EDGES)
        loss, p, t = train_step(edge_ids)
        losses.append(loss)
        print(f"epoch {epoch} step {step+1}/{STEPS_PER_EPOCH} | loss {loss:.4f} | batch pos {p}/{t}")
    print(f"epoch {epoch} DONE | avg loss {sum(losses)/len(losses):.4f}")

CSR ready: | nodes 5000000 | undirected edges 100000000 | avg deg 20.0 | neighbors GB 0.4
Pos edges: 63019 Neg edges: 49936981 pos_rate: 0.00126038
epoch 1 step 1/10 | loss 0.6977 | batch pos 100000/200000
epoch 1 step 2/10 | loss 0.6949 | batch pos 100000/200000
epoch 1 step 3/10 | loss 0.6821 | batch pos 100000/200000
epoch 1 step 4/10 | loss 0.6705 | batch pos 100000/200000
epoch 1 step 5/10 | loss 0.6648 | batch pos 100000/200000
epoch 1 step 6/10 | loss 0.6572 | batch pos 100000/200000
epoch 1 step 7/10 | loss 0.6467 | batch pos 100000/200000
epoch 1 step 8/10 | loss 0.6419 | batch pos 100000/200000
epoch 1 step 9/10 | loss 0.6353 | batch pos 100000/200000
epoch 1 step 10/10 | loss 0.6256 | batch pos 100000/200000
epoch 1 DONE | avg loss 0.6617
epoch 2 step 1/10 | loss 0.6217 | batch pos 100000/200000
epoch 2 step 2/10 | loss 0.6153 | batch pos 100000/200000
epoch 2 step 3/10 | loss 0.6087 | batch pos 100000/200000
epoch 2 step 4/10 | loss 0.6061 | batch pos 100000/200000
epoch 2 

In [8]:
import numpy as np
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

model.eval()

def predict_edges(edge_ids, fanout1=15, fanout2=10):
    s = src_np[edge_ids]
    d = dst_np[edge_ids]
    seed = np.unique(np.concatenate([s, d], axis=0)).astype(np.int32)

    e1_s, e1_d = sample_neighbors(seed, fanout1)
    hop1 = np.unique(np.concatenate([seed, e1_d], axis=0)).astype(np.int32)
    e2_s, e2_d = sample_neighbors(hop1, fanout2)
    local_nodes = np.unique(np.concatenate([hop1, e2_d], axis=0)).astype(np.int32)

    e1_src_l = map_global_to_local(local_nodes, e1_s)
    e1_dst_l = map_global_to_local(local_nodes, e1_d)
    e2_src_l = map_global_to_local(local_nodes, e2_s)
    e2_dst_l = map_global_to_local(local_nodes, e2_d)
    src_l = map_global_to_local(local_nodes, s)
    dst_l = map_global_to_local(local_nodes, d)

    local_nodes_t = torch.from_numpy(local_nodes.astype(np.int64)).to(device, non_blocking=True)
    e1_src_t = torch.from_numpy(e1_src_l).to(device)
    e1_dst_t = torch.from_numpy(e1_dst_l).to(device)
    e2_src_t = torch.from_numpy(e2_src_l).to(device)
    e2_dst_t = torch.from_numpy(e2_dst_l).to(device)
    src_tl = torch.from_numpy(src_l).to(device)
    dst_tl = torch.from_numpy(dst_l).to(device)

    xb = torch.from_numpy(X_edge_cpu[edge_ids]).to(device)

    with torch.no_grad():
        h = model.forward_subgraph(local_nodes_t, e1_src_t, e1_dst_t, e2_src_t, e2_dst_t)
        logits = model.edge_logits(h, src_tl, dst_tl, xb)
        prob = torch.sigmoid(logits).detach().cpu().numpy()
    return prob

VAL_EVAL_EDGES = 300_000
val_sample = rng.choice(val_e, size=min(VAL_EVAL_EDGES, len(val_e)), replace=False)

y_true = y_np[val_sample].astype(np.int32)
y_prob = predict_edges(val_sample)

print("VAL n:", len(y_true), "| pos:", int(y_true.sum()), "| pos_rate:", float(y_true.mean()))
print("VAL PR-AUC:", average_precision_score(y_true, y_prob))
print("VAL ROC-AUC:", roc_auc_score(y_true, y_prob))

VAL n: 300000 | pos: 398 | pos_rate: 0.0013266666666666667
VAL PR-AUC: 0.0037648731225316395
VAL ROC-AUC: 0.7786727963621017


In [9]:
!pip -q install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 24.7 MB/s eta 0:00:00


In [10]:
import numpy as np
import torch
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score, roc_auc_score

model.eval()

CB_TRAIN_N = 5_000_000
CB_VAL_N   = 500_000
CB_TEST_N  = 500_000

train_pick = rng.choice(train_e, size=min(CB_TRAIN_N, len(train_e)), replace=False)
val_pick   = rng.choice(val_e,   size=min(CB_VAL_N,   len(val_e)),   replace=False)
test_pick  = rng.choice(test_e,  size=min(CB_TEST_N,  len(test_e)),  replace=False)

def edge_embedding_batch(edge_ids, fanout1=15, fanout2=10):
    s = src_np[edge_ids]
    d = dst_np[edge_ids]
    seed = np.unique(np.concatenate([s, d], axis=0)).astype(np.int32)

    e1_s, e1_d = sample_neighbors(seed, fanout1)
    hop1 = np.unique(np.concatenate([seed, e1_d], axis=0)).astype(np.int32)
    e2_s, e2_d = sample_neighbors(hop1, fanout2)
    local_nodes = np.unique(np.concatenate([hop1, e2_d], axis=0)).astype(np.int32)

    e1_src_l = map_global_to_local(local_nodes, e1_s)
    e1_dst_l = map_global_to_local(local_nodes, e1_d)
    e2_src_l = map_global_to_local(local_nodes, e2_s)
    e2_dst_l = map_global_to_local(local_nodes, e2_d)
    src_l = map_global_to_local(local_nodes, s)
    dst_l = map_global_to_local(local_nodes, d)

    local_nodes_t = torch.from_numpy(local_nodes.astype(np.int64)).to(device, non_blocking=True)
    e1_src_t = torch.from_numpy(e1_src_l).to(device)
    e1_dst_t = torch.from_numpy(e1_dst_l).to(device)
    e2_src_t = torch.from_numpy(e2_src_l).to(device)
    e2_dst_t = torch.from_numpy(e2_dst_l).to(device)
    src_tl = torch.from_numpy(src_l).to(device)
    dst_tl = torch.from_numpy(dst_l).to(device)

    xb = torch.from_numpy(X_edge_cpu[edge_ids]).to(device)

    with torch.no_grad():
        h = model.forward_subgraph(local_nodes_t, e1_src_t, e1_dst_t, e2_src_t, e2_dst_t)
        hs = h[src_tl]
        hd = h[dst_tl]
        emb = torch.cat([hs, hd, hs * hd], dim=1).detach().cpu().numpy().astype(np.float32)

    feat = X_edge_cpu[edge_ids].astype(np.float32, copy=False)
    Xcb = np.concatenate([emb, feat], axis=1)  # [B, 3*out_dim + 12]
    ycb = y_np[edge_ids].astype(np.int32, copy=False)
    return Xcb, ycb

def build_cb_matrix(edge_ids, chunk=200_000):
    Xs, ys = [], []
    for i in range(0, len(edge_ids), chunk):
        sl = edge_ids[i:i+chunk]
        Xb, yb = edge_embedding_batch(sl)
        Xs.append(Xb); ys.append(yb)
        if (i//chunk + 1) % 5 == 0:
            print(f"  built {i+len(sl):,}/{len(edge_ids):,}")
    return np.vstack(Xs), np.concatenate(ys)

print("Building CatBoost train matrix...")
X_train_cb, y_train_cb = build_cb_matrix(train_pick, chunk=200_000)
print("Train matrix:", X_train_cb.shape, "pos:", y_train_cb.sum())

print("Building CatBoost val matrix...")
X_val_cb, y_val_cb = build_cb_matrix(val_pick, chunk=200_000)
print("Val matrix:", X_val_cb.shape, "pos:", y_val_cb.sum())

cb = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    verbose=100,
    random_seed=42
)

cb.fit(Pool(X_train_cb, y_train_cb), eval_set=Pool(X_val_cb, y_val_cb), use_best_model=True)

val_prob = cb.predict_proba(X_val_cb)[:, 1]
print("CatBoost VAL PR-AUC:", average_precision_score(y_val_cb, val_prob))
print("CatBoost VAL ROC-AUC:", roc_auc_score(y_val_cb, val_prob))

print("Building CatBoost test matrix...")
X_test_cb, y_test_cb = build_cb_matrix(test_pick, chunk=200_000)
test_prob = cb.predict_proba(X_test_cb)[:, 1]
print("CatBoost TEST PR-AUC:", average_precision_score(y_test_cb, test_prob))
print("CatBoost TEST ROC-AUC:", roc_auc_score(y_test_cb, test_prob))

Building CatBoost train matrix...
  built 1,000,000/5,000,000
  built 2,000,000/5,000,000
  built 3,000,000/5,000,000
  built 4,000,000/5,000,000
  built 5,000,000/5,000,000
Train matrix: (5000000, 204) pos: 6367
Building CatBoost val matrix...
Val matrix: (500000, 204) pos: 643
0:	learn: 0.5391559	test: 0.5391487	best: 0.5391487 (0)	total: 9.29s	remaining: 5h 9m 27s
100:	learn: 0.0086872	test: 0.0088917	best: 0.0088917 (100)	total: 11.8s	remaining: 3m 42s
200:	learn: 0.0084456	test: 0.0087631	best: 0.0087631 (200)	total: 14.3s	remaining: 2m 7s
300:	learn: 0.0082789	test: 0.0086946	best: 0.0086946 (300)	total: 16.7s	remaining: 1m 34s
400:	learn: 0.0081375	test: 0.0086414	best: 0.0086414 (400)	total: 19.1s	remaining: 1m 15s
500:	learn: 0.0080026	test: 0.0085997	best: 0.0085997 (500)	total: 21.4s	remaining: 1m 4s
600:	learn: 0.0078839	test: 0.0085637	best: 0.0085637 (600)	total: 23.8s	remaining: 55.4s
700:	learn: 0.0077783	test: 0.0085384	best: 0.0085384 (700)	total: 26.2s	remaining: 48.